In [6]:
!pip install pymupdf
!pip install sentence_transformers
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 95.1 MB/s eta 0:00:00


In [7]:
import pymupdf as fitz
import re
import torch
import faiss
from sentence_transformers import SentenceTransformer

In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embeddingmodel = SentenceTransformer('sentence-transformers/All-MiniLM-L6-v2', device = device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
def extract_text_from_pdf(pdf_path):
  pdf_file = fitz.open(pdf_path)
  text_data = []
  for page_number, page in enumerate(pdf_file, start = 1):
    text_content = page.get_text() or ""
    char_count = len(text_content)
    words = [word for word in re.split(" ", text_content) if word]
    word_count = len(words)
    page_data = {
        'text': text_content,
        'page_number': page_number,
        'word_count': word_count,
        'char_count': char_count
    }
    text_data.append(page_data)
  pdf_file.close()
  return(text_data)

In [10]:
def clean_text(text):

  if text is None:
    return ''

  # removing special tokens
  special_tokens = ["[CLS]", "[SEP]", "[PAD]", "[UNK]", "<s>", "</s>",
                    "<pad>", "</pad>", "<unk>"]
  for token in special_tokens:
    text = text.replace(token, ' ')

  # removing invisible utf chars
  invisible_chars = ["\u200b", "\u200c", "\u200d", "\ufeff", "\xa0"]
  for char in invisible_chars:
    text = text.replace(invisible_chars, ' ')

  # removing \r, \n and new line breaks
  text = re.sub(r'(\w)–\n(\w)', r'\1\2', text)
  text = text.replace('\n', ' ').replace('\r', ' ')

  # squash multiple spaces
  text = re.sub(r"\s+''", ' ', text)

  # fix space before punctuation
  text = re.sub(r"\s+[.,:;!?]", r'\1', text)

  # final strip
  return text.strip()

In [11]:
def split_into_sentences(text):
  sentences = re.split(r'(?<=[.,?])\s+(?=[A-Za-z0-9"\'\[(])', text)
  sentences = [sentence.strip() for sentence in sentences if sentence.strip()]
  return sentences

In [12]:
def chunk_text_with_metadata(cleaned_pages_data, max_words, sentence_overlap):
  final_chunk_dictionary = []
  sentence_batch = []
  chunk_id = 1
  word_count = 0
  for page in cleaned_pages_data:
    text = split_into_sentences(page['text'])
    for sentence in text:
      sentence_word = len(sentence.split())
      if sentence_batch and (word_count + sentence_word > max_words):
        chunk_text = " ".join(sentence_batch)
        final_chunk_dictionary.append({
          'chunk_id': chunk_id,
          'page_number': page['page_number'],
          'chunk_text': chunk_text
        })
        chunk_id = chunk_id + 1
        sentence_batch = sentence_batch[-sentence_overlap:]
        word_count = sum(len(s.split()) for s in sentence_batch)
      sentence_batch.append(sentence)
      word_count += sentence_word

    # accounting for leftover words, those who are left and didn't trigger
    # max_words count
    if sentence_batch:
      chunk_text = " ".join(sentence_batch)
      final_chunk_dictionary.append({
        'chunk_id': chunk_id,
        'page_number': page['page_number'],
        'chunk_text': chunk_text
      })
      chunk_id  = chunk_id + 1
      sentence_batch = []
      word_count = 0

  return final_chunk_dictionary

In [13]:
def create_chunk_embeddings(chunks):
  chunk_data = [chunk['chunk_text'] for chunk in chunks]
  embeddings = embeddingmodel.encode(chunk_data, convert_to_numpy=True, normalize_embeddings =True)
  return embeddings

In [14]:
def build_faiss_index(embeddings):
  dimension = embeddings.shape[1]    # getting shape
  index = faiss.IndexFlatIP(dimension)     # creating a faiss index based on our chunk shape
  index.add(embeddings)   # adding the embeddings into the fiass index
  return index

In [15]:
def query_embedding(query, embeddingmodel):
  # adding brackets around query to quickly pack it as a list.
  query_embed = embeddingmodel.encode([query], convert_to_numpy=True, normalize_embeddings= True)
  return query_embed

In [16]:
def faiss_search(query_embedding, index, top_value = 5):
  distances, indices = index.search(query_embedding, top_value)
  return distances, indices

In [17]:
def retrive_final_answers(indices, final_chunk_dictionary):
  top_chunk_ids = indices[0]
  retrieved_results = []

  for i in top_chunk_ids:
    retrieved_results.append(final_chunk_dictionary[i])

  return retrieved_results

## MAIN BLOCK

In [18]:
from google.colab import files

def run_retrival():
    # 1. Prompt the user to upload a PDF
    print("Please upload your PDF document:")
    uploaded = files.upload()

    # Grab the filename of the uploaded file
    pdf_filename = list(uploaded.keys())[0]
    print(f"\nSuccessfully uploaded: {pdf_filename}")

    # 2. Prompt the user for their question
    user_query = input("\nWhat would you like to ask about this document? ")

    # The RAG Pipeline

    # 3. Extract the raw text
    cleaned_pages_data = extract_text_from_pdf(pdf_filename)

    # 4. Chunk the text
    final_chunks = chunk_text_with_metadata(cleaned_pages_data, 250, 2)

    # 5. Embed the chunks
    chunk_embeddings = create_chunk_embeddings(final_chunks)

    # 6. Build the FAISS index
    index = build_faiss_index(chunk_embeddings)

    # 7. Embed the user's query
    # Note: embeddingmodel must be loaded in the cell above this!
    query_vector = query_embedding(user_query, embeddingmodel)

    # 8. Search the index
    distances, indices = faiss_search(query_vector, index, 3)

    # 9. Retrieve the final text dictionaries
    final_results = retrive_final_answers(indices, final_chunks)

    # Display the Results
    print(f"\n--- Top Results for: '{user_query}' ---\n")
    for i, chunk in enumerate(final_results):
        print(f"Result {i+1} (Page {chunk['page_number']}):")
        print(f"{chunk['chunk_text']}\n")

# Generation
commenting everthing

In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [20]:
# Load the raw tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
generator_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct").to(device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [21]:
def generate_final_answer_manual(query, retrieved_chunks, generator_model, tokenizer, tau=0.7, max_tokens=50):
    # 1. Prepare the text prompt
    context_string = " ".join([chunk['chunk_text'] for chunk in retrieved_chunks])
    prompt = f"Context: {context_string}\nQuestion: {query}\nAnswer:"

    # 2. Convert text to PyTorch tensors
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]

    # The Custom Generation Loop
    for _ in range(max_tokens):
        # Run the forward pass
        with torch.no_grad():
            outputs = generator_model(input_ids)

        # Grab the raw scores for the very last token in the sequence
        last_logits = outputs.logits[:, -1, :]

        # 3. Apply Temperature Scaling (tau) to the raw logits
        scaled_logits = last_logits / tau

        # 4. Convert those scaled scores into probabilities (0.0 to 1.0)
        probs = torch.softmax(scaled_logits, dim=-1)

        # 5. Sample the next token ID based on those probabilities
        next_token = torch.multinomial(probs, num_samples=1)

        # Append the new token to our running list
        input_ids = torch.cat([input_ids, next_token], dim=-1)

        # Stop early if the model decides it is finished
        if next_token.item() == tokenizer.eos_token_id:
            break

    # Decode the final array of numbers back into an English string
    final_text = tokenizer.decode(input_ids[0], skip_special_tokens=True, truncation=True, MAX_LENGTH=1000)
    return final_text

# GENERATION MAIN BLOCK

In [22]:
def run_complete_rag():
    # 1. Prompt the user to upload a PDF
    print("Please upload your PDF document:")
    uploaded = files.upload()

    # Grab the filename of the uploaded file
    pdf_filename = list(uploaded.keys())[0]
    print(f"\nSuccessfully uploaded: {pdf_filename}")

    # 2. Prompt the user for their question
    user_query = input("\nWhat would you like to ask about this document? ")

    # The RAG Pipeline

    # 3. Extract the raw text
    cleaned_pages_data = extract_text_from_pdf(pdf_filename)

    # 4. Chunk the text
    final_chunks = chunk_text_with_metadata(cleaned_pages_data, 250, 2)

    # 5. Embed the chunks
    chunk_embeddings = create_chunk_embeddings(final_chunks)

    # 6. Build the FAISS index
    index = build_faiss_index(chunk_embeddings)

    # 7. Embed the user's query
    query_vector = query_embedding(user_query, embeddingmodel)

    # 8. Search the index
    distances, indices = faiss_search(query_vector, index, 3)

    # 9. Retrieve the final text dictionaries
    final_results = retrive_final_answers(indices, final_chunks)

    ai_answer = generate_final_answer_manual(
        query=user_query,
        retrieved_chunks=final_results,
        generator_model=generator_model,
        tokenizer=tokenizer
    )

    # Display the Results
    print(ai_answer)

In [23]:
# run_complete_rag()

Please upload your PDF document:


KeyboardInterrupt: 

# CHATBOT CREATION

In [27]:
def generate_chat_answer_manual(query, retrieved_chunks, generator_model, tokenizer, history, tau=0.1, max_tokens=60):
    # 1. Prepare the context text
    context_string = " ".join([chunk['chunk_text'] for chunk in retrieved_chunks])

    # 2. Build the memory string from past interactions
    memory_string = ""
    for past_question, past_answer in history:
        memory_string += f"Question: {past_question}\nAnswer: {past_answer}\n"

    # 3. Inject the memory string into the final prompt
    prompt = f"Context: {context_string}\n{memory_string}Question: {query}\nAnswer:"

    # 4. Convert text to tensors
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=900).to(device)
    input_ids = inputs["input_ids"]

    # 5. The PyTorch Generation Loop
    for _ in range(max_tokens):
        with torch.no_grad():
            outputs = generator_model(input_ids)

        last_logits = outputs.logits[:, -1, :]
        scaled_logits = last_logits / tau
        probs = torch.softmax(scaled_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_token], dim=-1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    # 6. Decode and clean up the conversational dump
    final_text = tokenizer.decode(input_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    clean_text = final_text.split("Human:")[0].split("Question:")[0].split("\n")[0].strip()

    return clean_text

In [28]:
from google.colab import files

def start_chat():
    # 1. Upload and Process the PDF
    print("Please upload your PDF document to start the chat:")
    uploaded = files.upload()
    pdf_filename = list(uploaded.keys())[0]

    print(f"\nProcessing {pdf_filename}... Building vector index.")

    # 2. Ingest the document using your pre-defined functions
    cleaned_pages_data = extract_text_from_pdf(pdf_filename)
    final_chunks = chunk_text_with_metadata(cleaned_pages_data, 250, 2)
    chunk_embeddings = create_chunk_embeddings(final_chunks)
    index = build_faiss_index(chunk_embeddings)

    print("PDF processed successfully! Chatbot initialized. Type 'exit' to stop.\n")

    # 3. Initialize memory
    chat_history = []

    # 4. Enter the interactive loop
    while True:
        user_query = input("You: ")

        if user_query.lower() == 'exit':
            print("Ending chat. Goodbye!")
            break

        # 5. Embed the query
        query_vector = query_embedding(user_query, embeddingmodel)

        # 6. Retrieve relevant context
        distances, indices = faiss_search(query_vector, index, 3)
        final_results = retrive_final_answers(indices, final_chunks)

        # 7. Generate the conversational answer
        ai_answer = generate_chat_answer_manual(
            query=user_query,
            retrieved_chunks=final_results,
            generator_model=generator_model,
            tokenizer=tokenizer,
            history=chat_history
        )

        print(f"Bot: {ai_answer}\n")

        # 8. Save the interaction to memory
        chat_history.append((user_query, ai_answer))

In [31]:
start_chat()

Please upload your PDF document to start the chat:


Saving AIML NOTES.pdf to AIML NOTES (2).pdf

Processing AIML NOTES (2).pdf... Building vector index.
PDF processed successfully! Chatbot initialized. Type 'exit' to stop.

You: what is stride?
Bot: The stride controls how many pixels the kernel shifts per step; larger stride = smaller output, less computation, possible detail loss.

You: Okay, is that a part of some bigger topic?
Bot: Yes, it's part of the topic of convolutional neural networks (CNNs) and their applications in computer vision.

You: exit
Ending chat. Goodbye!
